In [1]:

# python mne-lsl-Trigger.py
import numpy as np

import pyxdf

from mne.io import read_raw_fif
from mne.time_frequency import psd_array_multitaper
from numpy.typing import NDArray
from scipy.integrate import simpson
from scipy.signal import periodogram, welch

# import mne_lsl
from mne_lsl.datasets import sample
from mne_lsl.player import PlayerLSL
from mne_lsl.stream import StreamLSL, EpochsStream

from mne_lsl.lsl import (
    StreamInfo,
    StreamInlet,
    StreamOutlet,
    local_clock,
    resolve_streams,
)



import mne
from mne.preprocessing import (ICA, create_eog_epochs, create_ecg_epochs, corrmap)
# import mne_connectivity

from hypyp import analyses

from asrpy import asr_calibrate, asr_process, clean_windows

import pathlib

import websockets
import asyncio

import time

from pythonosc.udp_client import SimpleUDPClient


c:\Users\thiag\anaconda3\Lib\site-packages\paramiko\pkey.py:82: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
c:\Users\thiag\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.Blowfish and will be removed from this module in 45.0.0.
  "class": algorithms.Blowfish,
c:\Users\thiag\anaconda3\Lib\site-packages\paramiko\transport.py:243: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "class": algorithms.TripleDES,


In [ ]:
freq_bands = {
    'Delta': [0.5,3.5],
    'Theta': [3.5,7],
    'Alpha':[7,14],
    'Beta': [14, 25]}

nChan = 16
sfreq = 125

from dataclasses import dataclass
@dataclass
class streamClass:
    EEGtimeSeries: np.ndarray
    TRGtimeSeries: np.ndarray
    timeSeries: np.ndarray
    timeStamps: np.ndarray
    SR: float


streamName_1 = "Cyton_COM10"
streamName_2 = "Cyton_COM11"

In [ ]:
ch_names = ['Fp1', 'Fp2', 'F3', 'F4', 'T3', 'C3', 'Cz', 'C4', 'T4', 'T5', 'P3', 'Pz', 'P4', 'T6', 'O1', 'O2']
ch_names_hyp = ['Fp1_1', 'Fp2_1', 'F3_1', 'F4_1', 'T3_1', 'C3_1', 'Cz_1', 'C4_1', 'T4_1', 'T5_1', 'P3_1', 'Pz_1', 'P4_1', 'T6_1', 'O1_1', 'O2_1', 'Fp1_2', 'Fp2_2', 'F3_2', 'F4_2', 'T3_2', 'C3_2', 'Cz_2', 'C4_2', 'T4_2', 'T5_2', 'P3_2', 'Pz_2', 'P4_2', 'T6_2', 'O1_2', 'O2_2']
ch_types_hyp = ['eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg']

subset_names = ['Fp1', 'Fp2', 'F3', 'F4', 'C3', 'C4', 'Cz', 'P3', 'P4', 'Pz']
subset_indices = [ch_names.index(name) for name in subset_names]


In [ ]:
print("Resolving LSL streams...")
bufferSize = 10 # seconds
nsamples = int(sfreq*bufferSize)

streams = resolve_streams()
print([s.name for s in streams])


# Retrive LSL streams in 1 second buffers
stream_lead = StreamLSL(bufsize=bufferSize, name="Cyton_COM10").connect()
stream_follow = StreamLSL(bufsize=bufferSize, name="Cyton_COM11").connect()

for ch in stream_lead.info['chs']:
    print(ch['ch_name'])

In [ ]:

epochs_lead = EpochsStream(
    stream_lead,
    bufsize=20,  # number of epoch held in the buffer
    event_id=256,
    event_channels="P11",
    tmin=-3,
    tmax=3,
    baseline=(None, 0),
    picks="eeg",
).connect(acquisition_delay=0.1)

epochs_follow = EpochsStream(
    stream_follow,
    bufsize=20,  # number of epoch held in the buffer
    event_id=256,
    event_channels="P11",
    tmin=-3,
    tmax=3,
    baseline=(None, 0),
    picks="eeg",
).connect(acquisition_delay=0.1)

In [ ]:
while epochs_lead.n_new_epochs < 5:
    time.sleep(5)

np.set_printoptions(threshold=5000)
print("events in P11: ", stream_lead.get_data(picks="P11")[0])
# print(stream_follow.get_data(picks="P11"))
print(f"Lead stream has {epochs_lead.n_new_epochs} epochs available.")
np.set_printoptions(threshold=100)

# Ensure the streams are connected
if not stream_lead.connected or not stream_follow.connected:
    raise RuntimeError("Failed to connect to the LSL streams.")